In [ ]:
!pip -q install -U transformers datasets accelerate sentence-transformers rank_bm25 scikit-learn evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.6 MB/s eta 0:00:00


In [ ]:
import os, re, json, math, random, zipfile
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

import numpy as np
from tqdm import tqdm

import torch
from torch.utils.data import Dataset

from rank_bm25 import BM25Okapi # Lexical retrieval baseline
from sentence_transformers import SentenceTransformer, CrossEncoder # Dense encoder and Cross encoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device : ",DEVICE)

def set_seed(seed: int = 42): # Define a single place to control randomness for comparable experiments
  random.seed(seed) # seed python rng (affects shuffling, random choice)
  np.random.seed(seed) # seed numpy rng (affects argsort tie-breaks)
  torch.manual_seed(seed) # seed pytorch cpu rng (affects dropout/init)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

CHUNK_SENTENCES = 3               # chunk size in sentences
CHUNK_STRIDE = 2                  # overlap stride in sentences
TOPK_BM25 = 40                     # candidate chunks from BM25
TOPK_DENSE = 40                    # candidate chunks from dense retriever
TOPK_RERANK = 12                   # final evidence chunks after rerank

DENSE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
VERIFIER_MODEL_NAME = "microsoft/deberta-v3-small"  # good verifier backbone

MAX_LEN = 256
BATCH_SIZE = 16
LR = 2e-5
EPOCHS = 2

print("Config loaded.")

Device :  cuda
Config loaded.


In [ ]:
ZIP_PATH = "/content/semeval2026-task12-dataset-main.zip"
EXTRACT_ROOT = "/content"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
  z.extractall(EXTRACT_ROOT)

DATA_DIR = "/content/semeval2026-task12-dataset-main/"

print("DATA_DIR : ", DATA_DIR)
print("Root contents : ", os.listdir(DATA_DIR))

DATA_DIR :  /content/semeval2026-task12-dataset-main
Root contents :  ['dev_data', 'test_data', 'sample_data', 'README.md', 'train_data', 'semeval2026-task12-dataset-old']


In [ ]:
def read_json(path):
  with open(path, "r", encoding="utf-8") as f:
    return json.load(f) # parse the .json file and return the python object

def read_jsonl(path):
  rows = [] # store the parsed rows here
  with open(path, "r", encoding="utf-8") as f:
    for line in f:
      line = line.strip() # remove whitespace/newline
      if line:
        rows.append(json.loads(line))
  return rows

def load_split(split_folder_name):

    split_dir = os.path.join(DATA_DIR, split_folder_name)

    q_path = os.path.join(split_dir, "questions.jsonl")
    d_path = os.path.join(split_dir, "docs.json")

    questions = read_jsonl(q_path)
    doc_rows = read_json(d_path)
    topic2docs = {}
    for r in doc_rows:
        topic2docs[r["topic_id"]] = r["docs"]

    merged = []

    for q in questions:

        topic_id = q["topic_id"]

        merged.append({
            "id": q.get("id"),
            "topic_id": topic_id,
            "target_event": q.get("target_event"),
            "options": {
                "A": q.get("option_A"),
                "B": q.get("option_B"),
                "C": q.get("option_C"),
                "D": q.get("option_D"),
            },
            "gold": q.get("golden_answer"),  # None for test
            "docs": topic2docs.get(topic_id, [])
        })

    return merged

train_data = load_split("train_data")
dev_data = load_split("dev_data")
test_data = load_split("test_data")

print("Loaded:", len(train_data), len(dev_data), len(test_data))
print("Sample train keys:", train_data[0].keys())
print("Sample event:", train_data[0]["target_event"])
print("Sample options:", train_data[0]["options"])
print("Sample docs count:", len(train_data[0]["docs"]))
print("Sample gold:", train_data[0]["gold"])

Loaded: 1819 400 612
Sample train keys: dict_keys(['id', 'topic_id', 'target_event', 'options', 'gold', 'docs'])
Sample event: Iran launched ballistic missile attacks against Al Asad and Erbil air bases in Iraq used by US and coalition forces on Tuesday night.
Sample options: {'A': 'On December 29, US forces conducted airstrikes at five Kataib Hezbollah facilities in Iraq and Syria, killing at least 25 people.', 'B': 'After 2006, Muhandis founded Kataib Hezbollah.', 'C': 'On December 27, Kataib Hezbollah attacked the K1 military base near Kirkuk, killing an American contractor and wounding American and Iraqi personnel.', 'D': 'A U.S. drone strike killed Iranian Maj. Gen. Qassem Soleimani and Abu Mahdi al-Muhandis after Muhandis picked up Soleimani from an airplane.'}
Sample docs count: 20
Sample gold: D


In [ ]:
SENT_SPLIT = re.compile(r"(?<=[\.\?\!])\s+(?=[A-Z0-9\"'])")

def split_sentences(text: str) -> List[str]:
  text = re.sub(r"\s+", " ", (text or "").strip())
  if not text:
    return []

  sents = SENT_SPLIT.split(text) # split text into sentences using the regex heuristic

  return [s.strip() for s in sents if len(s.strip()) > 0]

def chunk_sentences(
    sents: List[str],
    chunk_size: int = 3,
    stride: int = 2
) -> List[Tuple[str, Tuple[int, int]]]:
  """
  Convert a list of sentences into overlapping windows (chunks)
  Returns : List of (chunk_text, (Start_sent_index, end_sent_index_exclusive))
  """

  out = [] # store final chunks
  n = len(sents) # number of sentences available

  if n == 0:
    return out

  i = 0
  while i < n:
    j = min(n, i + chunk_size) # end index for this chunk dont exceed n
    chunk = " ".join(sents[i:j]).strip() # join the sentence window into one chunk

    if chunk:
      out.append((chunk, (i,j)))

    if j == n:
      break

    i += stride # move the window start by stride to create overlap

  return out


def doc_to_chunks(doc: Dict[str,str], doc_idx: int) -> List[Dict[str, Any]]:
  """
  Convert a single document dict fromm docs.json into chunk dicts
  """
  text = doc.get("content")
  title = doc.get("title")
  full = (title + ". " + text).strip() if title else text.strip() # prepend the sentence with title to help retrieval
  sents = split_sentences(full)
  chunks = chunk_sentences(sents, CHUNK_SENTENCES, CHUNK_STRIDE)

  out = []
  for ch_text, (a,b) in chunks:
    out.append({
        "chunk_text": ch_text,
        "doc_idx": doc_idx,
        "sent_span": (a,b),
        "title": title
    })

  return out

def build_topic_chunks(ex: Dict[str,Any]) -> List[Dict[str,Any]]:
  """
  Convert all docs in a topic into one flat list of chunk dicts.
  ex -> a topic-level dict with a key "docs" that is a list of documents.
  """

  chunks = []

  for i,d in enumerate(ex["docs"]):
    chunks.extend(doc_to_chunks(d,i)) # convert this doc into chunk dicts and add them to the topic list

  return chunks

chunks0 = build_topic_chunks(train_data[0])
print("chunks for sample topic:", len(chunks0))
print("first chunk:", chunks0[0]["chunk_text"][:200])

chunks for sample topic: 256
first chunk: Explosive drone downed near Kurdish Peshmerga forces in Iraq's Kirkuk province. Explosive drone downed near Kurdish Peshmerga forces in Iraq's Kirkuk province The Kurdistan Regional Government’s (KRG)


In [ ]:
def simple_tokenize(text: str) -> List[str]:
  return re.findall(r"[A-Za-z0-9]+", (text or "").lower()) # Extract alphanumeric tokens and lowercase them

dense_encoder = SentenceTransformer(DENSE_MODEL_NAME, device=DEVICE)

def build_retrieval_struct(ex: Dict[str, Any]):
  chunks = build_topic_chunks(ex)
  if len(chunks) == 0:
    return {
        "chunks": [],
        "bm25": None,
        "bm25_tokens": [],
        "dense_emb": None
    }

  tokenized = [simple_tokenize(c["chunk_text"]) for c in chunks]

  bm25 = BM25Okapi(tokenized)
  texts = [c["chunk_text"] for c in chunks]

  emb = dense_encoder.encode(
      texts,
      batch_size=64,
      show_progress_bar=False,
      convert_to_numpy=True,
      normalize_embeddings=True
  )

  return {
      "chunks": chunks,
      "bm25": bm25,
      "bm25_tokens": tokenized,
      "dense_emb": emb
  }

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
reranker = CrossEncoder(RERANK_MODEL_NAME, device=DEVICE)

def retrieve_evidence_for_option(ex: Dict[str,Any], retr: Dict[str,Any], option_text: str) -> List[Dict[str, Any]]:
  chunks = retr["chunks"]
  if not chunks:
    return []

  # Build query combining event + hypothesis
  query = (ex["target_event"] + " [HYPOTHESIS] " + option_text).strip()

  bm25 = retr["bm25"]
  bm25_scores = bm25.get_scores(simple_tokenize(query))
  bm25_idx = np.argsort(-bm25_scores)[:TOPK_BM25]

  q_emb = dense_encoder.encode(
      [query],
      convert_to_numpy=True,
      normalize_embeddings=True
  )[0]

  sims = retr["dense_emb"] @ q_emb
  dense_idx = np.argsort(-sims)[:TOPK_DENSE]

  cand_idx = sorted(set(bm25_idx.tolist()) | set(dense_idx.tolist()))
  cand_chunks = [chunks[i] for i in cand_idx]

  pairs = [(query, c["chunk_text"]) for c in cand_chunks]
  scores = reranker.predict(pairs, batch_size=64)

  order = np.argsort(-scores)[:TOPK_RERANK]

  top = []
  for j in order:
    c = dict(cand_chunks[int(j)]) # chunk dict
    c["rerank_score"] = float(scores[int(j)]) # attach rerank score
    top.append(c)

  return top

def evidence_to_text(evidence_chunks: List[Dict[str,Any]], max_chars: int = 2500) -> str:
  parts = []
  tot = 0

  for c in evidence_chunks:
    t = c["chunk_text"].strip()
    if not t:
      continue

    add = t + "\n"

    if tot + len(add) > max_chars: # truncate if exceeding max len
      break

    parts.append(add)
    tot += len(add)

  return "".join(parts).strip()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
NONE_PATTERN = re.compile(r"\bnone of the others\b|\bnone of the above\b", re.I)

def find_none_label(options: Dict[str,str]) -> str:
  """
  Identify which option corresponds to "none".
  """

  for k,v in options.items():
    if v and NONE_PATTERN.search(v):
      return k
  return None

def parse_gold(gold: str) -> List[str]:
  """
  Parse comma-seperated gold labels
  """
  if gold is None:
    return []
  gold = gold.strip()
  if not gold:
    return []
  return [x.strip() for x in gold.split(",") if x.strip()]

def build_rows(split_data: List[Dict[str, Any]], cache: Dict[int, Any] = None) -> List[Dict[str, Any]]:
  rows = []
  if cache is None:
    cache = {}

  for ex in tqdm(split_data, desc="build_rows"):
    topic_id = ex["topic_id"]
    if topic_id not in cache:
      cache[topic_id] = build_retrieval_struct(ex)

    retr = cache[topic_id]

    none_label = find_none_label(ex["options"])
    gold_labels = set(parse_gold(ex["gold"]))

    for opt_k, opt_text in ex["options"].items():
      if opt_text is None:
          continue

      evidence_chunks = retrieve_evidence_for_option(ex, retr, opt_text)
      ev_text = evidence_to_text(evidence_chunks)

      label = None
      if ex["gold"] is not None:
          label = 1 if opt_k in gold_labels else 0

      rows.append({
          "id": ex["id"],
          "topic_id": topic_id,
          "event": ex["target_event"],
          "option_key": opt_k,
          "option_text": opt_text,
          "none_key": none_label,
          "evidence": ev_text,
          "label": label
      })
  return rows

train_rows = build_rows(train_data)
dev_rows   = build_rows(dev_data)

print("Train rows:", len(train_rows), "Dev rows:", len(dev_rows))
print(train_rows[0].keys())

build_rows: 100%|██████████| 400/400 [07:46<00:00,  1.17s/it]

Train rows: 7276 Dev rows: 1600
dict_keys(['id', 'topic_id', 'event', 'option_key', 'option_text', 'none_key', 'evidence', 'label'])


In [ ]:
ver_tokenizer = AutoTokenizer.from_pretrained(VERIFIER_MODEL_NAME)

def format_input(event: str, option_text: str, evidence: str) -> str:
  """
  Structured prompt for verified model
  """
  return (
      f"EVENT: {event}\n"
      f"HYPOTHESIS (direct cause): {option_text}\n"
      f"EVIDENCE: \n{evidence}"
  )

class VerifierDataset(Dataset):

  def __init__(self, rows, tokenizer, max_len=256, train=True):
    self.rows = rows
    self.tok = tokenizer
    self.max_len = max_len
    self.train = train

  def __len__(self):
    return len(self.rows)

  def __getitem__(self, idx):
    r = self.rows[idx]

    text = format_input(
        r["event"],
        r["option_text"],
        r["evidence"]
    )

    enc = self.tok(
        text,
        truncation=True,
        max_length=self.max_len,
        padding="max_length"
    )

    item = {k: torch.tensor(v) for k, v in enc.items()}

    if self.train:
      item["labels"] = torch.tensor(
          int(r["label"]),
          dtype=torch.long
      )

    return item

train_ds = VerifierDataset(train_rows, ver_tokenizer, MAX_LEN, train=True)
dev_ds   = VerifierDataset(dev_rows, ver_tokenizer, MAX_LEN, train=True)

print("Example keys:", train_ds[0].keys())

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Example keys: dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])


In [ ]:
def decode_from_probs(option_probs, none_key, thr):
  """
  From the probabilities for the options choose the one with highest
  """

  keys = ["A","B","C","D"]
  chosen = []

  for k in keys:
    if k == none_key:
      continue
    if option_probs.get(k, 0.0) >= thr:
      chosen.append(k)

  if len(chosen) == 0:
    return none_key if none_key is not None else "D"

  return ",".join(chosen)


def semeval_score(gold, pred):
  """
  Calculate the score of the results
  """

  G = set(parse_gold(gold))
  P = set(parse_gold(pred))
  if P == G:
      return 1.0
  if len(P) > 0 and P.issubset(G) and P != G:
      return 0.5
  return 0.0

def evaluate_split(examples: List[Dict[str,Any]], pred_map: Dict[Any,str]) -> float:
  scores = []
  for ex in examples:
      pid = ex["id"]
      scores.append(semeval_score(ex["gold"], pred_map[pid]))
  return float(np.mean(scores))

def build_pred_map_from_probs(split_examples: List[Dict[str,Any]], probs_map: Dict[Any,Dict[str,float]], thr: float) -> Dict[Any,str]:
    pred_map = {}
    for ex in split_examples:
        qid = ex["id"]
        none_key = find_none_label(ex["options"])
        pred_map[qid] = decode_from_probs(probs_map[qid], none_key, thr)
    return pred_map

def evaluate_with_breakdown(examples: List[Dict[str,Any]], pred_map: Dict[Any,str]):
    exact = subset = wrong = 0
    for ex in examples:
        s = semeval_score(ex["gold"], pred_map[ex["id"]])
        if s == 1.0:
            exact += 1
        elif s == 0.5:
            subset += 1
        else:
            wrong += 1
    n = len(examples)
    mean_score = (exact*1.0 + subset*0.5 + wrong*0.0) / n
    breakdown = {
        "exact": exact/n,
        "subset": subset/n,
        "wrong": wrong/n
    }
    return mean_score, breakdown

In [ ]:
from transformers import DataCollatorWithPadding

In [ ]:
ver_model = AutoModelForSequenceClassification.from_pretrained(
    VERIFIER_MODEL_NAME,
    num_labels=2
).to(DEVICE)

training_args = TrainingArguments(
    output_dir = os.path.join(OUT_DIR, f"verifier_seed{SEED}"),
    learning_rate = LR,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = BATCH_SIZE,
    num_train_epochs = EPOCHS,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    fp16=False,
    bf16=True,
    report_to = "none",
)

data_collator = DataCollatorWithPadding(tokenizer=ver_tokenizer)

trainer = Trainer(
    model = ver_model,
    args = training_args,
    train_dataset = train_ds,
    eval_dataset = dev_ds,
    data_collator=data_collator,
)

trainer.train()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Epoch,Training Loss,Validation Loss
1,No log,0.676628
2,0.769949,0.673300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

TrainOutput(global_step=910, training_loss=0.7257072029532967, metrics={'train_runtime': 702.1592, 'train_samples_per_second': 20.725, 'train_steps_per_second': 1.296, 'total_flos': 963867125096448.0, 'train_loss': 0.7257072029532967, 'epoch': 2.0})

In [ ]:
@torch.no_grad()
def predict_option_probs(rows, tokenizer, model):
  model.eval()
  out = {}
  bs = BATCH_SIZE

  for i in tqdm(range(0, len(rows), bs)):
    batch = rows[i:i+bs]

    texts = [
        format_input(r["event"],
                     r["option_text"],
                     r["evidence"])
        for r in batch
    ]

    enc = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
        return_tensors="pt"
    ).to(DEVICE)

    logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1)[:, 1]

    for r,p in zip(batch, probs):
      qid = r["id"]
      if qid not in out:
        out[qid] = {}
      out[qid][r["option_key"]] = float(p.cpu())

  return out

dev_probs = predict_option_probs(dev_rows, ver_tokenizer, ver_model)
print("One dev probs:", next(iter(dev_probs.items())))

100%|██████████| 100/100 [00:23<00:00,  4.30it/s]

One dev probs: ('q-2020', {'A': 0.3882700502872467, 'B': 0.3882700502872467, 'C': 0.3882700502872467, 'D': 0.3882700502872467})


In [ ]:
def row_level_metrics(rows, probs_map, thr=0.5):
    ys, ps = [], []
    for r in rows:
        if r["label"] is None:
            continue
        ys.append(int(r["label"]))
        ps.append(float(probs_map[r["id"]][r["option_key"]]))
    ys = np.array(ys); ps = np.array(ps)

    auc = roc_auc_score(ys, ps) if len(np.unique(ys)) > 1 else float("nan")
    yhat = (ps >= thr).astype(int)
    acc = accuracy_score(ys, yhat)
    pr, rc, f1, _ = precision_recall_fscore_support(ys, yhat, average="binary", zero_division=0)
    return {"auc": auc, "acc": acc, "prec": pr, "rec": rc, "f1": f1}

print("Row metrics @thr=0.5:", row_level_metrics(dev_rows, dev_probs, thr=0.5))

Row metrics @thr=0.5: {'auc': 0.5, 'acc': 0.6, 'prec': 0.0, 'rec': 0.0, 'f1': 0.0}


In [ ]:
best_thr, best_score = None, -1

for thr in np.linspace(0.1, 0.9, 17):
    pred_map = build_pred_map_from_probs(dev_data, dev_probs, float(thr))
    sc = evaluate_split(dev_data, pred_map)
    if sc > best_score:
        best_score, best_thr = sc, float(thr)

In [ ]:
pred_map = build_pred_map_from_probs(dev_data, dev_probs, best_thr)
score, breakdown = evaluate_with_breakdown(dev_data, pred_map)
print("DEV SemEval:", score)
print("Breakdown:", breakdown)
print("BEST THR:", best_thr, "DEV SCORE:", best_score)
print("Row metrics @best_thr:", row_level_metrics(dev_rows, dev_probs, thr=best_thr))

DEV SemEval: 0.37125
Breakdown: {'exact': 0.2425, 'subset': 0.2575, 'wrong': 0.5}
BEST THR: 0.4 DEV SCORE: 0.37125
Row metrics @best_thr: {'auc': 0.5, 'acc': 0.6, 'prec': 0.0, 'rec': 0.0, 'f1': 0.0}


In [ ]:
def thr_sweep_table(dev_data, dev_probs, thrs):
    out = []
    for thr in thrs:
        pred_map = build_pred_map_from_probs(dev_data, dev_probs, float(thr))
        score, breakdown = evaluate_with_breakdown(dev_data, pred_map)
        out.append((thr, score, breakdown["exact"], breakdown["subset"], breakdown["wrong"]))
    return out

table = thr_sweep_table(dev_data, dev_probs, np.linspace(0.1, 0.9, 9))
for thr, score, exa, sub, wr in table:
    print("thr=%.2f score=%.4f exact=%.3f subset=%.3f wrong=%.3f" % (thr, score, exa, sub, wr))

thr=0.10 score=0.0000 exact=0.000 subset=0.000 wrong=1.000
thr=0.20 score=0.0000 exact=0.000 subset=0.000 wrong=1.000
thr=0.30 score=0.0000 exact=0.000 subset=0.000 wrong=1.000
thr=0.40 score=0.3713 exact=0.242 subset=0.258 wrong=0.500
thr=0.50 score=0.3713 exact=0.242 subset=0.258 wrong=0.500
thr=0.60 score=0.3713 exact=0.242 subset=0.258 wrong=0.500
thr=0.70 score=0.3713 exact=0.242 subset=0.258 wrong=0.500
thr=0.80 score=0.3713 exact=0.242 subset=0.258 wrong=0.500
thr=0.90 score=0.3713 exact=0.242 subset=0.258 wrong=0.500


In [ ]:
def is_none_pred(pred, none_key):
    return pred.strip() == (none_key or "")

def analyse_none_errors(dev_data, pred_map):
    c = {"gold_none_pred_none":0, "gold_none_pred_non":0, "gold_non_pred_none":0, "gold_non_pred_non":0}
    for ex in dev_data:
        none_key = find_none_label(ex["options"])
        gold_set = set(parse_gold(ex["gold"]))
        gold_is_none = (none_key is not None and gold_set == {none_key})
        pred_is_none = is_none_pred(pred_map[ex["id"]], none_key)

        if gold_is_none and pred_is_none: c["gold_none_pred_none"] += 1
        elif gold_is_none and not pred_is_none: c["gold_none_pred_non"] += 1
        elif (not gold_is_none) and pred_is_none: c["gold_non_pred_none"] += 1
        else: c["gold_non_pred_non"] += 1
    n = len(dev_data)
    return {k: v/n for k,v in c.items()}

pred_map_best = build_pred_map_from_probs(dev_data, dev_probs, best_thr)
print("None-analysis:", analyse_none_errors(dev_data, pred_map_best))


None-analysis: {'gold_none_pred_none': 0.115, 'gold_none_pred_non': 0.0, 'gold_non_pred_none': 0.0, 'gold_non_pred_non': 0.885}


In [ ]:
def show_failures(dev_data, probs_map, thr, max_show=10):
    pred_map = build_pred_map_from_probs(dev_data, probs_map, thr)
    bad = []
    for ex in dev_data:
        pid = ex["id"]
        sc = semeval_score(ex["gold"], pred_map[pid])
        if sc < 1.0:
            bad.append((sc, ex, pred_map[pid], probs_map[pid]))
    bad.sort(key=lambda x: x[0])  # worst first

    for sc, ex, pred, pr in bad[:max_show]:
        print("="*80)
        print("ID:", ex["id"], "SemEval:", sc)
        print("EVENT:", ex["target_event"])
        print("GOLD:", ex["gold"], "PRED:", pred)
        print("PROBS:", {k: round(v,3) for k,v in pr.items()})
        print("OPTIONS:")
        for k,v in ex["options"].items():
            print(" ", k, ":", v[:160] if v else v)

show_failures(dev_data, dev_probs, best_thr, max_show=20)

ID: q-2020 SemEval: 0.0
EVENT: President Yoon Suk Yeol vowed to carry out a thorough investigation.
GOLD: C PRED: D
PROBS: {'A': 0.388, 'B': 0.388, 'C': 0.388, 'D': 0.388}
OPTIONS:
  A : Kim and her friend crawled out of the crush and were pulled into a tavern by adults.
  B : Emergency workers and pedestrians performed CPR on victims in the streets on Saturday night.
  C : At least 153 people were killed and dozens injured.
  D : Kim and her friend entered the alley at 8 p.m. and became trapped as the crowd density increased.
ID: q-2021 SemEval: 0.0
EVENT: Twitter removed Trump's tweets and locked his account.
GOLD: C PRED: D
PROBS: {'A': 0.388, 'B': 0.388, 'C': 0.388, 'D': 0.388}
OPTIONS:
  A : A woman was shot inside the Capitol and died.
  B : Pipe bombs were reported at the Republican National Committee building and the Democratic National Committee headquarters.
  C : Supporters stormed the U.S. Capitol.
  D : Capitol Police Chief Steven Sund, House Sergeant at Arms Paul Irving, 

In [ ]:
bad_id = "q-2027"
ex = next(e for e in dev_data if e["id"] == bad_id)
retr = build_retrieval_struct(ex)

print("EVENT:", ex["target_event"])
print("GOLD:", ex["gold"])
print("OPTIONS:", ex["options"])

for k, opt_text in ex["options"].items():
    ev_chunks = retrieve_evidence_for_option(ex, retr, opt_text)
    print("\n", "="*80)
    print("Option", k, ":", opt_text)
    for i, ch in enumerate(ev_chunks[:5]):
        print(f"\n  [chunk {i}] rerank_score={ch.get('rerank_score', None)}")
        print(" ", ch["chunk_text"][:500])

EVENT: Huawei released a TV running HarmonyOS.
GOLD: A,B
OPTIONS: {'A': 'Huawei unveiled its Harmony operating system.', 'B': 'Huawei unveiled its Harmony operating system.', 'C': "Huawei denied the U.S. government's accusations and pursued legal means.", 'D': 'Huawei announced HarmonyOS will be open source.'}

Option A : Huawei unveiled its Harmony operating system.

  [chunk 0] rerank_score=6.4852399826049805
  Honor Vision TV is Huawei’s first HarmonyOS product. Huawei recently unveiled HarmonyOS which it said was from a vision from over 10 years ago where they envision an interconnected world from all aspects. Being that the launch of their OS comes at a time where the company is facing a ban from America, and potential complete restriction from using Operating Systems like Android by Google, and Windows by Microsoft, most people are seeing Huawei’s announcement as a way of them showing the world the

  [chunk 1] rerank_score=6.365617275238037
  Huawei’s Hongmeng may not replace An

In [ ]:
test_rows = build_rows(test_data)
test_probs = predict_option_probs(test_rows,ver_tokenizer,ver_model)

test_pred_map = {}

for ex in test_data:
    qid = ex["id"]
    none_key = find_none_label(ex["options"])
    test_pred_map[qid] = decode_from_probs(
        test_probs[qid],
        none_key,
        best_thr
    )

sub_path = os.path.join(OUT_DIR, f"submission_seed{SEED}.jsonl")

with open(sub_path, "w", encoding="utf-8") as f:
    for ex in test_data:
        f.write(json.dumps({
            "id": ex["id"],
            "answer": test_pred_map[ex["id"]]
        }) + "\n")


100%|██████████| 153/153 [00:36<00:00,  4.24it/s]


In [ ]:
dev_prob_path = os.path.join(OUT_DIR, f"dev_probs_seed{SEED}.json")
test_prob_path = os.path.join(OUT_DIR, f"test_probs_seed{SEED}.json")

with open(dev_prob_path, "w", encoding="utf-8") as f:
    json.dump(dev_probs, f, ensure_ascii=False)

with open(test_prob_path, "w", encoding="utf-8") as f:
    json.dump(test_probs, f, ensure_ascii=False)

zip_path = os.path.join(OUT_DIR, f"seed{SEED}_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(dev_prob_path, arcname=os.path.basename(dev_prob_path))
    z.write(test_prob_path, arcname=os.path.basename(test_prob_path))

print("Saved:", zip_path)

Saved: outputs/seed42_outputs.zip
